In [4]:
# =============================================================================
# BLOCK 1: SETUP, IMPORTS, AND DATA LOADING
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import time
# --- Library Imports ---
import pandas as pd
import numpy as np
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import catboost as cb
import optuna
print("Libraries imported successfully.")
# --- Helper Function for Winkler Score ---
def winkler_score(y_true, lower, upper, alpha=0.1, return_coverage=False):
    width = upper - lower
    penalty_lower = np.where(y_true < lower, (2 / alpha) * (lower - y_true), 0)
    penalty_upper = np.where(y_true > upper, (2 / alpha) * (y_true - upper), 0)
    score = width + penalty_lower + penalty_upper
    if return_coverage:
        coverage = np.mean((y_true >= lower) & (y_true <= upper))
        return np.mean(score), coverage
    return np.mean(score)
# --- Global Constants ---
N_SPLITS = 5
RANDOM_STATE = 42
DATA_PATH = './'
N_OPTUNA_TRIALS = 30 # A strong number for a comprehensive search
COMPETITION_ALPHA = 0.1

# --- Load Raw Data ---
try:
    # We drop the low-variance columns they identified right away
    drop_cols=['id', 'golf', 'view_rainier', 'view_skyline', 'view_lakesamm','view_otherwater', 'view_other']
    df_train = pd.read_csv(DATA_PATH + 'dataset.csv').drop(columns=drop_cols)
    df_test = pd.read_csv(DATA_PATH + 'test.csv').drop(columns=drop_cols)
    print("Raw data loaded successfully.")
except FileNotFoundError:
    print("ERROR: Could not find 'dataset.csv' or 'test.csv'.")
    exit()
# --- Prepare Target Variable ---
y_true = df_train['sale_price'].copy()
# The mean-error model works best when predicting the raw price directly
# So, we will NOT log-transform the target this time.
# df_train.drop('sale_price', axis=1, inplace=True) # We keep sale_price for FE
print("Setup complete.")


Libraries imported successfully.
Raw data loaded successfully.
Setup complete.


In [5]:
# =============================================================================
# BLOCK 2: SYNTHESIZED FEATURE ENGINEERING (CORRECTED)
# =============================================================================
print("--- Starting Block 2: Synthesized Feature Engineering ---")
def create_synthesized_features(df_train, df_test):
    # Combine for consistent processing and reset the index
    df_train['is_train'] = 1
    df_test['is_train'] = 0
    # Store the original id for later, as reset_index will remove it
    train_ids = df_train.index
    test_ids = df_test.index
    all_data = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
    
    # --- A) Brute-Force Numerical Interactions ---
    print("Creating brute-force numerical interaction features...")
    NUMS = ['area', 'land_val', 'imp_val', 'sqft_lot', 'sqft', 'sqft_1','grade', 'year_built']
    for i in range(len(NUMS)):
        for j in range(i + 1, len(NUMS)):
            all_data[f'{NUMS[i]}_x_{NUMS[j]}'] = all_data[NUMS[i]] *all_data[NUMS[j]]
    
    # --- B) Date Features ---
    all_data['sale_date'] = pd.to_datetime(all_data['sale_date'])
    all_data['year'] = all_data['sale_date'].dt.year
    all_data['month'] = all_data['sale_date'].dt.month
    all_data['year_diff'] = all_data['year'] - all_data['year_built']
    
    # --- C) TF-IDF Text Features ---
    print("Creating TF-IDF features for text columns...")
    text_cols = ['subdivision', 'zoning', 'city', 'sale_warning','join_status', 'submarket']
    all_data[text_cols] = all_data[text_cols].fillna('missing').astype(str)
    for col in text_cols:
        tfidf = TfidfVectorizer(analyzer='char', ngram_range=(3, 5),max_features=128, binary=True)
        tfidf_matrix = tfidf.fit_transform(all_data[col])
        svd = TruncatedSVD(n_components=8, random_state=RANDOM_STATE)
        tfidf_svd = svd.fit_transform(tfidf_matrix)
        tfidf_df = pd.DataFrame(tfidf_svd, columns=[f'{col}_tfidf_svd_{i}' for i in range(8)])
        
        # This concat will now work because both have a simple 0-based index
        all_data = pd.concat([all_data, tfidf_df], axis=1)
    
    # --- D) Log transform some of the new interaction features ---
    for c in ['land_val_x_imp_val', 'land_val_x_sqft', 'imp_val_x_sqft']:
        if c in all_data.columns:
            # Add a small constant to avoid log(0)
            all_data[c] = np.log1p(all_data[c].fillna(0))
    
    # --- E) Final Cleanup ---
    print("Finalizing feature set...")
    cols_to_drop = ['sale_date', 'subdivision', 'zoning', 'city','sale_warning', 'join_status', 'submarket']
    all_data = all_data.drop(columns=cols_to_drop)
    all_data.fillna(0, inplace=True)
    
    # Separate final datasets
    X = all_data[all_data['is_train'] == 1].drop(columns=['is_train','sale_price'])
    X_test = all_data[all_data['is_train'] == 0].drop(columns=['is_train','sale_price'])
    
    # Restore the original 'id' as the index
    X.index = train_ids
    X_test.index = test_ids
    X_test = X_test[X.columns]
    return X, X_test

# We need to re-run this from the original dataframes
X, X_test = create_synthesized_features(df_train, df_test)
print(f"\nSynthesized FE complete. Total features: {X.shape[1]}")
gc.collect()


--- Starting Block 2: Synthesized Feature Engineering ---
Creating brute-force numerical interaction features...
Creating TF-IDF features for text columns...
Finalizing feature set...

Synthesized FE complete. Total features: 111


245

In [6]:
# =======================================================================================
#
# STAGE 1: TRAIN THE CATBOOST MEAN MODEL
#
# =======================================================================================
print("--- STAGE 1: K-Fold Training of CatBoost Mean Model ---")
print("# Using the optimal hyperparameters found previously.")

# The best parameters you found for the CatBoost mean model
best_params_catboost = {
    'iterations': 2329,
    'learning_rate': 0.07106828663102695,
    'depth': 9,
    'l2_leaf_reg': 0.011456642952301135,
    'subsample': 0.901161247633512,
    'random_strength': 0.6138817518014036,
    'bagging_temperature': 0.017313451474763708,
    'random_seed': RANDOM_STATE,
    'verbose': 0
}

# Initialize arrays to store predictions
oof_catboost_preds = np.zeros(len(X))
test_catboost_preds = np.zeros(len(X_test))

# Setup K-Fold splits
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
grade_for_stratify = pd.read_csv(DATA_PATH + 'dataset.csv')['grade']

for fold, (train_idx, val_idx) in enumerate(skf.split(X, grade_for_stratify)):
    print(f"  Training CatBoost Mean Model - Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_true.iloc[train_idx], y_true.iloc[val_idx]
    
    model = cb.CatBoostRegressor(**best_params_catboost)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              early_stopping_rounds=100,
              verbose=0)
    
    oof_catboost_preds[val_idx] = model.predict(X_val)
    test_catboost_preds += model.predict(X_test) / N_SPLITS
    
# --- Evaluate Mean Model Performance ---
final_mean_rmse_cb = np.sqrt(mean_squared_error(y_true, oof_catboost_preds))
print("\n--- CatBoost K-Fold Training Complete & Performance Metrics ---")
print(f"CatBoost Final OOF RMSE: ${final_mean_rmse_cb:,.2f}")

--- STAGE 1: K-Fold Training of CatBoost Mean Model ---
# Using the optimal hyperparameters found previously.
  Training CatBoost Mean Model - Fold 1/5...
  Training CatBoost Mean Model - Fold 2/5...
  Training CatBoost Mean Model - Fold 3/5...
  Training CatBoost Mean Model - Fold 4/5...
  Training CatBoost Mean Model - Fold 5/5...

--- CatBoost K-Fold Training Complete & Performance Metrics ---
CatBoost Final OOF RMSE: $98,246.71


In [8]:
# =======================================================================================
#
# STAGE 2: PREPARE AND TRAIN THE ERROR MODEL
#
# =======================================================================================

# --- PART A: Prepare Data & Tune Hyperparameters ---
print("\n--- STAGE 2, PART A: Tuning the Error Model on CatBoost's Errors ---")

# 1. Create the new error target and feature set
# (This assumes these variables are already created from Cell 3)
# error_target_catboost = np.abs(y_true - oof_catboost_preds)
# X_for_error_cb = X.copy()
# X_for_error_cb['mean_pred_oof'] = oof_catboost_preds

# 2. Define Optuna objective function for the error model
def create_error_objective(X_features, y_error):
    X_train, X_val, y_train, y_val = train_test_split(X_features, y_error, test_size=0.25, random_state=RANDOM_STATE)
    def objective(trial):
        params = {
            'objective': 'reg:squarederror','eval_metric': 'rmse','tree_method': 'hist',
            'eta': trial.suggest_float('eta', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 10),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'lambda': trial.suggest_float('lambda', 1e-8, 10.0, log=True),
            'alpha': trial.suggest_float('alpha', 1e-8, 10.0, log=True),
            'seed': RANDOM_STATE, 'n_jobs': -1
        }
        model = xgb.XGBRegressor(**params, n_estimators=2000, early_stopping_rounds=50)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        preds = model.predict(X_val)
        return np.sqrt(mean_squared_error(y_val, preds))
    return objective

# 3. Run the Optuna study
study_error_cb = optuna.create_study(direction='minimize')
objective_func_error = create_error_objective(X_for_error_cb, error_target_catboost)

# THE FIX IS HERE: Use study.optimize() instead of study.run()
study_error_cb.optimize(objective_func_error, n_trials=N_OPTUNA_TRIALS)

# 4. Store the best parameters
best_params_error_cb = study_error_cb.best_params
print("\n--- Error Model Tuning Complete ---")
print(f"Best validation RMSE found: ${study_error_cb.best_value:,.2f}")
print("Best hyperparameters for the new error model:", best_params_error_cb)

[I 2025-07-16 15:30:15,728] A new study created in memory with name: no-name-cabd9c68-bd95-4fef-892c-ac453312be14



--- STAGE 2, PART A: Tuning the Error Model on CatBoost's Errors ---


[I 2025-07-16 15:30:22,919] Trial 0 finished with value: 62918.69984595756 and parameters: {'eta': 0.01584290048213737, 'max_depth': 6, 'subsample': 0.5066841582267829, 'colsample_bytree': 0.6376814655318479, 'lambda': 5.143359500751453e-07, 'alpha': 7.769114192830539e-08}. Best is trial 0 with value: 62918.69984595756.
[I 2025-07-16 15:30:30,676] Trial 1 finished with value: 62876.80617841188 and parameters: {'eta': 0.013465484069933099, 'max_depth': 8, 'subsample': 0.6074532778902557, 'colsample_bytree': 0.7130222511143671, 'lambda': 0.004629924196521838, 'alpha': 0.00017399399102656506}. Best is trial 1 with value: 62876.80617841188.
[I 2025-07-16 15:30:35,790] Trial 2 finished with value: 62731.15353250014 and parameters: {'eta': 0.03783523357239864, 'max_depth': 9, 'subsample': 0.951989374121068, 'colsample_bytree': 0.5653297245641524, 'lambda': 0.010648324600520502, 'alpha': 0.01818916141831582}. Best is trial 2 with value: 62731.15353250014.
[I 2025-07-16 15:30:38,702] Trial 3 f


--- Error Model Tuning Complete ---
Best validation RMSE found: $62,485.03
Best hyperparameters for the new error model: {'eta': 0.012739055699272025, 'max_depth': 10, 'subsample': 0.9752141540437247, 'colsample_bytree': 0.5015211252982051, 'lambda': 1.643788957391447e-06, 'alpha': 1.480976151644356e-07}


In [9]:
# --- PART B: K-Fold Training of the Tuned Error Model ---
print("\n--- STAGE 2, PART B: K-Fold Training with Newly Tuned Parameters ---")

# Add fixed parameters to the tuned ones
final_params_error_cb = best_params_error_cb.copy()
final_params_error_cb.update({'objective': 'reg:squarederror', 'eval_metric': 'rmse',
                              'tree_method': 'hist', 'random_state': RANDOM_STATE, 'n_jobs': -1})

# Initialize prediction arrays
oof_error_preds_cb = np.zeros(len(X))
test_error_preds_cb = np.zeros(len(X_test))
X_test_for_error_cb = X_test.copy()
X_test_for_error_cb['mean_pred_oof'] = test_catboost_preds

for fold, (train_idx, val_idx) in enumerate(skf.split(X_for_error_cb, grade_for_stratify)):
    print(f"  Training Error Model - Fold {fold+1}/{N_SPLITS}...")
    X_train, X_val = X_for_error_cb.iloc[train_idx], X_for_error_cb.iloc[val_idx]
    y_train, y_val = error_target_catboost.iloc[train_idx], error_target_catboost.iloc[val_idx]

    model = xgb.XGBRegressor(**final_params_error_cb, n_estimators=2000, early_stopping_rounds=100)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    oof_error_preds_cb[val_idx] = model.predict(X_val)
    test_error_preds_cb += model.predict(X_test_for_error_cb) / N_SPLITS

final_error_rmse_cb = np.sqrt(mean_squared_error(error_target_catboost, oof_error_preds_cb))
print(f"\nNew Tuned Error Model Final OOF RMSE: ${final_error_rmse_cb:,.2f}")


--- STAGE 2, PART B: K-Fold Training with Newly Tuned Parameters ---
  Training Error Model - Fold 1/5...
  Training Error Model - Fold 2/5...
  Training Error Model - Fold 3/5...
  Training Error Model - Fold 4/5...
  Training Error Model - Fold 5/5...

New Tuned Error Model Final OOF RMSE: $62,709.68


In [11]:
# =======================================================================================
#
# STAGE 3: FINAL CALIBRATION AND SHOWDOWN
#
# =======================================================================================
print("\n" + "="*60)
print("             THE MOMENT OF TRUTH: FINAL SHOWDOWN")
print("="*60)

oof_error_final_cb = np.clip(oof_error_preds_cb, 0, None)
best_score_cb = float('inf')
best_a_cb, best_b_cb = 1.0, 1.0

print("Starting final calibration grid search for the fully-tuned CatBoost pipeline...")
for a in np.arange(1.90, 2.31, 0.01):
    for b in np.arange(2.10, 2.51, 0.01):
        low = oof_catboost_preds - oof_error_final_cb * a
        high = oof_catboost_preds + oof_error_final_cb * b
        score = winkler_score(y_true, low, high, alpha=COMPETITION_ALPHA)
        if score < best_score_cb:
            best_score_cb = score
            best_a_cb, best_b_cb = a, b

# THE FIX IS HERE: Remove the comma from the number definition
OLD_BEST_SCORE = 301553.47

print("\n--- FINAL RESULTS ---")
print(f"Original XGBoost Pipeline Final Score: {OLD_BEST_SCORE:,.2f}")
print(f"New CatBoost Pipeline Final Score    : {best_score_cb:,.2f}")
print(f"  (Using optimal multipliers a={best_a_cb:.2f}, b={best_b_cb:.2f})")

if best_score_cb < OLD_BEST_SCORE:
    print("\nCONCLUSION: VICTORY! The fully-tuned CatBoost-based pipeline is the new champion!")
else:
    print("\nCONCLUSION: Close! The original XGBoost pipeline remains superior.")


             THE MOMENT OF TRUTH: FINAL SHOWDOWN
Starting final calibration grid search for the fully-tuned CatBoost pipeline...

--- FINAL RESULTS ---
Original XGBoost Pipeline Final Score: 301,553.47
New CatBoost Pipeline Final Score    : 302,350.04
  (Using optimal multipliers a=1.95, b=2.18)

CONCLUSION: Close! The original XGBoost pipeline remains superior.
